# Modelos Avanzados de ML para Deteccion de Impago

Objetivo: mejorar las metricas del modelo base (FICO) y del Random Forest usando modelos avanzados de gradient boosting.

**Metricas de referencia (sobre test):**

| Metrica | Modelo Base (FICO) | Random Forest (05_) |
|---|---|---|
| Accuracy | 0.72 | 0.80 |
| AUC-ROC | 0.592 | 0.683 |
| Recall (Default) | 0.24 | 0.02 |
| F1 (Default) | 0.25 | 0.04 |

Modelos a probar: **CatBoost, LightGBM, XGBoost, HistGradientBoosting**

## PASO 1: Preprocesamiento y Filtrado (mismo pipeline que 05/06)

In [2]:
from src.preprocessing.base_preprocessing import BasePreprocess
from src.filtering.base_filtering import BaseFiltering

# Preprocesamiento
base_pre = BasePreprocess("data/variables_withoutExperts.xlsx", "loan_status")
base_pre.fit("data/df_train_small.csv")

# Transform train y test
X_train, y_train = base_pre.transform("data/df_train_small.csv")
X_test, y_test = base_pre.transform("data/df_test_small.csv")

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:62: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.train_X_data['earliest_cr_line'] = pd.to_datetime(self.train_X_data['earliest_cr_line'])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:135: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X_data['earliest_cr_line'] = pd.to_datetime(X_data['earliest_cr_line'])
/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:135: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X_data['earliest_cr

Train: (80000, 2614), Test: (20000, 2614)


In [3]:
# Filtrado de features
base_filter = BaseFiltering(
    constant_tol=0.9,
    correlation_threshold=0.8,
    probe_n_probes=10,
    probe_scoring='roc_auc',
    probe_cv=3,
    probe_n_estimators=50,
    probe_max_depth=10
)
base_filter.fit(X_train, y_train)
base_filter.print_summary()

X_train_filtered = base_filter.transform(X_train)
X_test_filtered = base_filter.transform(X_test)

print(f"\nTrain filtrado: {X_train_filtered.shape}")
print(f"Test filtrado:  {X_test_filtered.shape}")

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs

RESUMEN DEL PIPELINE DE FILTRADO
  Features iniciales:              2614
  Eliminadas cuasi-constantes:     -133
  Eliminadas por correlacion:       -1840
  Eliminadas por ProbeFeature:      -399
  Features seleccionadas finales:  242

Train filtrado: (80000, 242)
Test filtrado:  (20000, 242)


In [4]:
X_test_filtered.head()

,verification_status_Not Verified,verification_status_Verified,term_ 36 months,emp_title_00,emp_title_01,emp_title_02,emp_title_04,emp_title_05,emp_title_06,emp_title_07,...,mths_since_recent_bc num_accts_ever_120_pd,num_accts_ever_120_pd total_bc_limit,num_actv_rev_tl tot_hi_cred_lim,num_actv_rev_tl total_bc_limit,num_bc_sats tot_hi_cred_lim,num_bc_sats earliest_cr_line_year,num_bc_tl num_il_tl,num_sats total_bc_limit,num_sats earliest_cr_line_year,num_tl_op_past_12m total_bc_limit
0,0.0,1.0,1.0,-0.017448,0.155599,0.285106,0.558374,0.372511,0.180468,-0.229400,...,10.469757,-4.465761,3.224467,1.924830,2.006481,-0.380826,1.622952,1.627856,-0.517575,1.260759
1,0.0,0.0,0.0,-0.088001,-0.094741,0.387118,0.110625,0.043007,0.036960,0.152519,...,-6.630728,-2.647513,-0.082430,-0.414695,-0.052263,0.328602,-0.272560,-0.307020,0.383705,-2.647513
2,0.0,0.0,1.0,-0.085707,-0.138750,0.631284,-0.431350,0.227354,-0.010348,-0.149757,...,-2.850057,-2.995208,0.251458,0.194429,0.690045,0.020917,0.166656,0.268914,0.010542,-2.995208
3,0.0,0.0,0.0,0.477474,-0.506376,-0.354385,0.172951,0.132443,-0.024205,0.023791,...,-5.464111,4.845633,-0.090080,0.346923,-0.423537,-3.228828,-0.202915,0.815805,-1.614862,-0.088996
4,0.0,1.0,0.0,0.309946,-0.180818,-0.091186,0.211222,-0.022481,-0.404333,-0.322874,...,-0.095847,-0.985543,-0.018460,-0.661395,0.031801,0.137968,0.088432,-0.664629,-0.080480,-0.656556


In [5]:
import numpy as np

# Preparar arrays planos
y_train_flat = y_train.values.ravel()
y_test_flat = y_test.values.ravel()

# Calcular scale_pos_weight para modelos que lo necesitan (ratio negativo/positivo)
n_negative = np.sum(~y_train_flat)
n_positive = np.sum(y_train_flat)
scale_pos_weight = n_negative / n_positive
print(f"Clase mayoritaria (Fully Paid): {n_negative}")
print(f"Clase minoritaria (Default):    {n_positive}")
print(f"Ratio (scale_pos_weight):       {scale_pos_weight:.2f}")

Clase mayoritaria (Fully Paid): 63764
Clase minoritaria (Default):    16236
Ratio (scale_pos_weight):       3.93


In [6]:
# Funcion de evaluacion comun para todos los modelos
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

def evaluate_model(model, X_test, y_test, model_name, results_dict):
    """Evalua un modelo y guarda metricas en results_dict."""
    class_pred = model.predict(X_test)
    prob_pred = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, class_pred)
    prec = precision_score(y_test, class_pred)
    rec = recall_score(y_test, class_pred)
    f1 = f1_score(y_test, class_pred)
    auc = roc_auc_score(y_test, prob_pred)

    results_dict[model_name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'F1-Score': f1, 'AUC-ROC': auc,
        'prob_pred': prob_pred, 'class_pred': class_pred
    }

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(classification_report(y_test, class_pred,
                                target_names=["Fully Paid", "Default"]))
    print(f"  AUC-ROC: {auc:.4f}")
    return results_dict

# Diccionario para almacenar resultados
results = {}

In [7]:
X_test_filtered.shape

(20000, 242)